# Forecast robuste - Depots comptes cheques et comptes courants

Ce notebook est construit comme outil de prise de decision, pas comme generateur de prediction magique.

Principe:
- on reserve les 60 derniers jours comme hold-out final;
- la selection des modeles se fait avant le hold-out, via walk-forward rolling origin;
- les modeles complexes sont compares a des baselines simples;
- si le gain vs meilleur baseline est < 5%, le baseline est retenu;
- la projection 90 jours inclut un intervalle empirique et un niveau de fiabilite.


## Configuration

In [ ]:
from pathlib import Path

import pandas as pd

from robust_forecast_engine import ForecastConfig, load_workbook, run_bundle


config = ForecastConfig(
    input_path=Path("in/reserve_in.xlsx"),
    output_dir=Path("out"),
    holdout_len=60,
    forecast_horizon=90,
    wf_folds=5,
    wf_horizon=30,
    min_gain_vs_baseline_pct=5.0,
    peak_quantile=0.80,
    peak_metric_weight=0.45,
    peak_weight_multiplier=4.0,
    max_mape_degradation_vs_baseline_pct=25.0,
)

targets = [
    "Depots Clientele_comptes chèques",
    "Depots Clientele_comptes courants",
]


## Inspection rapide du fichier source

In [ ]:
df = load_workbook(config)
print(df.shape)
print(df.index.min(), "->", df.index.max())
display(df[targets].describe().T)


## Execution de la pipeline robuste

In [ ]:
results = run_bundle(
    name="depots_cheques_courants",
    targets=targets,
    config=config,
    add_total=True,
)


## Synthese decisionnelle

In [ ]:
summary_rows = []
for result in results:
    diag = result.diagnostics.set_index("metric")["value"]
    summary_rows.append({
        "serie": result.target,
        "modele_retenu": diag["final_model"],
        "modele_propose": diag["proposed_model"],
        "modele_scenario_pic": diag["peak_scenario_model"],
        "meilleur_baseline": diag["best_baseline"],
        "meilleur_baseline_mape": diag["best_mape_baseline"],
        "mape_holdout": diag["holdout_MAPE"],
        "peak_wmape_holdout": diag["holdout_peak_wMAPE"],
        "peak_capture": diag["holdout_peak_capture"],
        "score_decision": diag["holdout_score"],
        "selection_gain_vs_baseline_pct": diag["selection_gain_vs_best_baseline_pct"],
        "selection_mape_degradation_pct": diag["selection_mape_degradation_vs_best_baseline_pct"],
        "robustesse": diag["robustness_flag"],
        "recommendation": diag["decision_recommendation"],
    })
summary = pd.DataFrame(summary_rows)
display(summary)


## Projections 90 jours

In [ ]:
for result in results:
    print("\n", result.target)
    display(result.projection.head(15))


## Comparaison des modeles

In [ ]:
for result in results:
    print("\n", result.target)
    cols = [
        "model", "holdout_score", "holdout_MAPE", "holdout_peak_wMAPE",
        "holdout_peak_capture", "wf_selection_score", "wf_score_median", "wf_peak_wMAPE_median",
        "wf_peak_capture_mean", "ensemble_weight"
    ]
    display(result.candidates[cols].sort_values("holdout_score"))


## Audit multi-holdout sans fuite

In [ ]:
for result in results:
    print("\n", result.target)
    display(result.multi_holdout)
